# 🗂️ Notebook 2: Stock Exchange — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Order** — side (buy/sell), price, qty, trader.
- **Trade** — a matched pair.
- **Book** — per-symbol sorted price levels.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, PositiveInt

class Order(BaseModel):
    id: int
    symbol: str
    side: Literal["buy","sell"]
    price: float
    qty: PositiveInt
    trader: str

class Trade(BaseModel):
    buy_id: int
    sell_id: int
    symbol: str
    price: float
    qty: PositiveInt

o = Order(id=1, symbol="AAPL", side="buy", price=100.0, qty=10, trader="alice")
print(o)

## HTTP APIs

| Method | Path | What |
|---|---|---|
| POST | `/orders` | Submit new order |
| DELETE | `/orders/{id}` | Cancel |
| GET | `/book/{symbol}` | Top N levels |
| WS | `/feed/{symbol}` | Streaming trades + quotes |


## Quick demo

In [ ]:
# Tiny order book: list of bids (desc price) + asks (asc price)
import bisect
bids, asks = [], []  # elements: (price, id)

def add_buy(price, oid):
    bisect.insort(bids, (-price, oid))   # desc via negative key

def add_sell(price, oid):
    bisect.insort(asks, (price, oid))

add_buy(100, 1); add_buy(101, 2); add_sell(102, 3); add_sell(103, 4)
print("best bid:", -bids[0][0], " best ask:", asks[0][0])

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.